# Generative Models in Deep Learning
Generative models learn the underlying data distribution P(X) to create new, realistic samples. This notebook covers Autoencoders (AE), Variational Autoencoders (VAE), GANs, DCGAN, CycleGAN, and Diffusion Models.

## 1. Autoencoders (AE)
An Autoencoder compresses input into a low-dimensional bottleneck then reconstructs it.
- **Encoder**: z = f_enc(x) — maps input to latent code z
- **Decoder**: x_hat = f_dec(z) — reconstructs input from z
- **Loss**: ||x - x_hat||^2 (MSE) or Binary Cross-Entropy

Uses: dimensionality reduction, denoising, anomaly detection (high reconstruction error = anomaly).

In [1]:
from tensorflow.keras import layers, models
import tensorflow as tf

latent_dim = 32

enc_input = layers.Input(shape=(784,))
enc = layers.Dense(256, activation='relu')(enc_input)
z = layers.Dense(latent_dim)(enc)
encoder = models.Model(enc_input, z, name="Encoder")

dec_input = layers.Input(shape=(latent_dim,))
dec = layers.Dense(256, activation='relu')(dec_input)
dec_out = layers.Dense(784, activation='sigmoid')(dec)
decoder = models.Model(dec_input, dec_out, name="Decoder")

ae_input = layers.Input(shape=(784,))
ae_out = decoder(encoder(ae_input))
autoencoder = models.Model(ae_input, ae_out, name="Autoencoder")
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')
encoder.summary()

Model: "Encoder"

┏━━━━━━━━━┳━━━━━━━┳━━━━┓
┃ Layer   ┃ Outp… ┃ P… ┃
┃ (type)  ┃ Shape ┃  # ┃
┡━━━━━━━━━╇━━━━━━━╇━━━━┩
│ input_… │ (Non… │  0 │
│ (Input… │ 784)  │    │
├─────────┼───────┼────┤
│ dense   │ (Non… │ 2… │
│ (Dense) │ 256)  │    │
├─────────┼───────┼────┤
│ dense_1 │ (Non… │ 8… │
│ (Dense) │ 32)   │    │
└─────────┴───────┴────┘

 Total params: 209,184 (817.12 KB)

 Trainable params: 209,184 (817.12 KB)

 Non-trainable params: 0 (0.00 B)

## 2. Variational Autoencoders (VAE)
VAE adds a probabilistic prior to the latent space. The encoder outputs mean (mu) and log variance (log_var).

**Reparameterization Trick**: z = mu + sigma * epsilon, where epsilon ~ N(0,1)
This makes z differentiable w.r.t. mu and sigma, enabling backpropagation through sampling.

**Total VAE Loss = Reconstruction Loss + KL Divergence**
The KL term pulls the learned distribution toward N(0,I), creating a smooth, continuous latent space.

In [2]:
class Sampling(layers.Layer):
    # Reparameterization trick: z = mu + sigma * epsilon
    def call(self, inputs):
        mu, log_var = inputs
        epsilon = tf.random.normal(shape=tf.shape(mu))
        return mu + tf.exp(0.5 * log_var) * epsilon

enc_in = layers.Input(shape=(784,))
h = layers.Dense(256, activation='relu')(enc_in)
mu = layers.Dense(latent_dim)(h)
log_var = layers.Dense(latent_dim)(h)
z = Sampling()([mu, log_var])
vae_encoder = models.Model(enc_in, [mu, log_var, z], name="VAE_Encoder")
print("VAE Encoder outputs: mu, log_var, z")
vae_encoder.summary()

VAE Encoder outputs: mu, log_var, z


Model: "VAE_Encoder"

┏━━━━━━┳━━━━━┳━━┳━━━━━━┓
┃ Lay… ┃ Ou… ┃  ┃ Con… ┃
┃ (ty… ┃ Sh… ┃  ┃ to   ┃
┡━━━━━━╇━━━━━╇━━╇━━━━━━┩
│ inp… │ (N… │  │ -    │
│ (In… │ 78… │  │      │
├──────┼─────┼──┼──────┤
│ den… │ (N… │  │ inp… │
│ (De… │ 25… │  │      │
├──────┼─────┼──┼──────┤
│ den… │ (N… │  │ den… │
│ (De… │ 32) │  │      │
├──────┼─────┼──┼──────┤
│ den… │ (N… │  │ den… │
│ (De… │ 32) │  │      │
├──────┼─────┼──┼──────┤
│ sam… │ (N… │  │ den… │
│ (Sa… │ 32) │  │ den… │
└──────┴─────┴──┴──────┘

 Total params: 217,408 (849.25 KB)

 Trainable params: 217,408 (849.25 KB)

 Non-trainable params: 0 (0.00 B)

## 3. Generative Adversarial Networks (GANs)
Proposed by Ian Goodfellow (2014). Two networks in a minimax game:
- **Generator G**: Noise z → fake data G(z)
- **Discriminator D**: Real or fake data → P(real)

```
Objective: min_G max_D [ E[log D(x)] + E[log(1 - D(G(z)))] ]
```

## 4. DCGAN
Uses CNNs in both G and D:
- G: Dense → Reshape → ConvTranspose2D (upsample) → Tanh output
- D: Conv2D (downsample, stride=2) → LeakyReLU → Dense
- BatchNormalization in both; NO pooling layers

## 5. CycleGAN
Unpaired image-to-image translation (e.g., horse → zebra).
Uses two generators (G: X→Y, F: Y→X) and cycle-consistency loss: G(F(y)) ≈ y, F(G(x)) ≈ x

## 6. Diffusion Models (DDPM)
State-of-the-art generative quality. Two processes:
- **Forward**: Gradually adds Gaussian noise over T steps until x_T is pure noise
- **Reverse**: a U-Net learns to predict and remove the noise at each step t

Training: minimize ||epsilon - epsilon_theta(sqrt(alpha_bar_t)*x0 + sqrt(1-alpha_bar_t)*epsilon, t)||^2

In [3]:
def make_generator(latent_dim=100):
    return models.Sequential([
        layers.Dense(7*7*256, use_bias=False, input_shape=(latent_dim,)),
        layers.BatchNormalization(), layers.LeakyReLU(),
        layers.Reshape((7, 7, 256)),
        layers.Conv2DTranspose(128, 5, strides=1, padding='same', use_bias=False),
        layers.BatchNormalization(), layers.LeakyReLU(),
        layers.Conv2DTranspose(64, 5, strides=2, padding='same', use_bias=False),
        layers.BatchNormalization(), layers.LeakyReLU(),
        layers.Conv2DTranspose(1, 5, strides=2, padding='same',
                               use_bias=False, activation='tanh'),
    ], name="DCGAN_Generator")

def make_discriminator():
    return models.Sequential([
        layers.Conv2D(64, 5, strides=2, padding='same', input_shape=(28,28,1)),
        layers.LeakyReLU(), layers.Dropout(0.3),
        layers.Conv2D(128, 5, strides=2, padding='same'),
        layers.LeakyReLU(), layers.Dropout(0.3),
        layers.Flatten(), layers.Dense(1)
    ], name="DCGAN_Discriminator")

G = make_generator()
D = make_discriminator()
print("Generator params:", G.count_params())
print("Discriminator params:", D.count_params())

Generator params: 2330944
Discriminator params: 212865


C:\AI-Projects\MyAI\DL\dl_env_conda\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\AI-Projects\MyAI\DL\dl_env_conda\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


# Conclusions and Key Takeaways
- **AE**: Fast, interpretable compression — but not truly generative (no meaningful prior to sample from)
- **VAE**: Probabilistic and generative; latent space is smooth and structured for interpolation and sampling
- **GAN**: Sharpest image quality but training is adversarially unstable and prone to mode collapse
- **Diffusion**: Currently the highest quality generative models, with stable training but slow iterative inference

# Pros and Cons
**Pros:**
- VAE: Stable optimization, smooth latent manifold, principled Bayesian framework
- GAN: Fastest sampling at inference, photorealistic image quality, diverse architectures (DCGAN, StyleGAN, CycleGAN)
- Diffusion: State-of-the-art FID scores, stable training, flexible conditioning (text, class, image)

**Cons:**
- VAE: Blurry reconstructions because pixel-wise MSE optimizes for the average, not sharpness
- GAN: Mode collapse, vanishing generator gradients, brittle hyperparameters
- Diffusion: Hundreds of sequential denoising steps → very slow inference (mitigated by DDIM, consistency models)

# 15 Interview Questions and Answers

1. **What is the bottleneck in an Autoencoder?**
   *Answer*: A hidden layer with fewer neurons than the input, forcing the network to learn only the most informative compressed representation of the data.

2. **What is the Reparameterization Trick?**
   *Answer*: Instead of sampling z ~ N(mu, sigma^2) directly (non-differentiable), we write z = mu + sigma * epsilon where epsilon ~ N(0,1). This makes z a deterministic, differentiable function of mu and sigma, enabling backprop through the sampling step.

3. **What is KL Divergence in the VAE loss?**
   *Answer*: It measures how much the learned posterior q(z|x) diverges from the standard normal prior N(0,I). Minimizing it regularizes the latent space to be continuous and prevents encodings from being arbitrarily scattered.

4. **What is Mode Collapse in GANs?**
   *Answer*: The Generator produces only a small subset of possible outputs — always generating the same few images that successfully fool D. It ignores the full diversity of the real data distribution.

5. **Why is GAN training unstable?**
   *Answer*: G and D are simultaneously optimizing competing objectives in a dynamic minimax game. If D becomes too strong, G receives zero gradient. If G gets too good, D collapses. There is no guaranteed convergence to a stable Nash equilibrium.

6. **What is a Wasserstein GAN?**
   *Answer*: Uses Earth Mover's distance instead of Jensen-Shannon divergence as the loss, providing continuous, meaningful gradients even when G and D distributions don't overlap, stabilizing training significantly.

7. **How do Conditional GANs work?**
   *Answer*: Both G and D receive class label (or other conditioning) as additional input. G generates class-conditional samples: G(z, y); D verifies that a sample matches both the class y and real data distribution.

8. **What is the Forward Process in DDPM?**
   *Answer*: A fixed Markov chain that progressively adds small amounts of Gaussian noise to a data point x_0 over T steps (e.g., T=1000), until x_T is indistinguishable from random Gaussian noise.

9. **What does the neural network in DDPM predict?**
   *Answer*: It predicts the noise epsilon that was added to create x_t from x_0. The reverse process then uses this prediction to subtract the estimated noise and recover x_{t-1}.

10. **Why do diffusion models produce sharper images than VAEs?**
    *Answer*: VAEs minimize pixel-wise MSE, which averages plausible images, causing blurriness. Diffusion models iteratively refine in noise space using a powerful U-Net with attention, implicitly learning a perceptual loss.

11. **What is a Latent Diffusion Model (LDM)?**
    *Answer*: Performs the diffusion process in the compressed latent space of a pretrained VAE encoder rather than pixel space, dramatically reducing computational cost. Stable Diffusion is built on this architecture.

12. **What is Frechet Inception Distance (FID)?**
    *Answer*: The primary metric for evaluating generative models. It computes the Frechet distance between the distribution of Inception features of real vs. generated images. Lower FID = better quality and diversity.

13. **What is StyleGAN?**
    *Answer*: A GAN where a mapping network converts latent codes to style vectors that control AdaIN normalization layers throughout the generator, allowing fine-grained control over coarse-to-fine visual attributes.

14. **What is DDIM and why is it useful?**
    *Answer*: Denoising Diffusion Implicit Models reformulates the reverse process as a non-Markovian process, allowing much fewer denoising steps (50 instead of 1000) at inference time while maintaining quality.

15. **How does CycleGAN achieve unpaired image translation?**
    *Answer*: It trains two generators (G: X->Y, F: Y->X) and two discriminators. The cycle consistency loss enforces F(G(x)) = x and G(F(y)) = y, ensuring translations are meaningful without requiring paired training images.
